# 24 — Subclonal evolution + TCR/co-stimulation signaling (on the re-annotation)

Consolidates the old `old/23_subclonal_evolution` (subclonal evolution) + `old/26_tcr_signaling_subclones` (TCR-signaling subclones) onto the **new**
re-annotation `skin_T_annotated.h5ad` (`21_reannotation`) and the v4 malignancy outputs from `23_malignancy_tcr_cnv`
(`alice_malignancy_v3`, `skin_T_malignancy_v4`, `skin_T_arm_cnv_v4`). Malignant compartment =
`tcr_malignant_alice` (ALICE dominant founder + ≤1-aa β-variant family). Helpers reused verbatim
(`subclone_helpers`, `alice_helpers`, `tcr_signaling_helpers`).

- **§A** — subclonal (divergent-evolution) structure: per-donor CNV subclones, transcriptomic
  MAJOR → CNV MINOR nested subclones, trunk/branch arm events, functional programs, tropism.
- **§B** — TCR / co-stimulation signaling: where in the TCR cascade each malignant clone/subclone
  concentrates dysregulation, vs lineage-matched reactive-CD4, cross-checked against arm-CNV.

> **HEAVY** — loads/normalizes the ~18 GB object; run on the GPU/compute kernel, not the login node.
> The subclone detection itself is CPU-light off the cached arm-CNV matrix.

# §A — Subclonal (divergent-evolution) analysis

In [ ]:
# ============================================================
# §A.0  Parameters
# ============================================================
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "4")
import sys
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); print("NB_DIR =", NB_DIR)
OUT_DIR = NB_DIR / "data" / "atlas_joint"
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

# ---- inputs (new re-annotation + nb30 v4 malignancy outputs) ----
OBJ           = OUT_DIR / "skin_T_annotated.h5ad"               # nb10b re-annotation (carries X_mrvi_u, raw_counts)
ALICE_MAL     = OUT_DIR / "alice_malignancy_v3.parquet"         # nb30: tcr_malignant_alice per cell
MALIG_PARQUET = OUT_DIR / "skin_T_malignancy_v4.parquet"        # nb30: cnv_arm_malignant + tcr_* columns
ARM           = OUT_DIR / "skin_T_arm_cnv_v4.parquet"           # nb30: per-cell 41-arm inferCNV matrix + held-out diploid null
GMT           = NB_DIR.parent / "lib1_immune.gmt"

# ---- knobs (as nb23) ----
SEED = 0
MIN_MAL = 200
K_MAX = 4
MIN_SUB = 50
MIN_LAYER_CELLS = 30
# subclone gates: calibrated against the held-out diploid null (nb30 v4 arm cache carries it),
# not hard-coded. SIL_MIN / MIN_ARM_DELTA are the fallback for a null-less matrix, never a floor.
SIL_MIN = 0.15
MIN_ARM_DELTA = 0.03
NULL_Z = 3.0
NULL_DRAWS = 8
Z_VLIM = 3.0        # arm-profile colour limit in diploid SDs (fixed, not self-normalizing)
Z_THR, EPS = 2.5, 0.01
BRANCH_DELTA = 0.03
AUC_THR, SIL_THR = 0.65, 0.10
LATENT = "X_mrvi_u"
LEIDEN_RES = 0.2
N_NEIGHBORS = 15

# ---- outputs (v3) ----
SUBCLONE_PARQUET = OUT_DIR / "subclones_v3.parquet"
SUMMARY_CSV      = OUT_DIR / "subclone_summary_v3.csv"
TRUNKBRANCH_CSV  = OUT_DIR / "subclone_trunk_branch_v3.csv"
ELIG_CSV         = OUT_DIR / "subclone_donor_eligibility_v3.csv"
SIGN_JSON        = OUT_DIR / "subclone_signatures_v3.json"
DE_DIR           = OUT_DIR / "subclone_de"; DE_DIR.mkdir(exist_ok=True)

In [ ]:
import importlib, gc
import numpy as np, pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sys.path.insert(0, str(NB_DIR / "helpers"))
import subclone_helpers as S
import alice_helpers as A
import skin_T_cnv_helpers as H
for _m in (S, A, H):
    importlib.reload(_m)
np.random.seed(SEED)
sc.settings.verbosity = 1
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ============================================================
# §A.0  Load: malignant compartment + cached arm-CNV matrix
# ============================================================
if not os.environ.get("LSB_JOBID"):
    print("⚠️  No LSB_JOBID detected — you may be on the login node; loading skin_T_annotated.h5ad (~18 GB) can crash it.")
adata = sc.read_h5ad(OBJ)
print("loaded", OBJ.name, adata.shape)
# X and 'counts' are unused (rebuilt from raw_counts in §A.9 / §B); free them to keep peak RAM low.
adata.X = None
if "counts" in adata.layers:
    del adata.layers["counts"]
# new re-annotation carries cell_type_T (CD4/CD8/Tregs/UNK); alias to cell_type_T2 for the shared helpers.
adata.obs["cell_type_T2"] = adata.obs["cell_type_T"].astype(str)

# Malignancy = TCR-ALICE only (nb30). Join tcr_malignant_alice + the TCR/CNV columns from the parquets.
_al = pd.read_parquet(ALICE_MAL).set_index("cell_id")
adata.obs["tcr_malignant_alice"] = (_al["tcr_malignant_alice"].reindex(adata.obs_names)
                                    .fillna(False).astype(bool).to_numpy())
assert MALIG_PARQUET.exists(), f"{MALIG_PARQUET} missing — run 30_malignant_annotation_tcr_cnv.ipynb first"
_mp = pd.read_parquet(MALIG_PARQUET)
_bool = {"cnv_arm_malignant", "tcr_is_dominant_clone", "tcr_is_expanded", "tcr_is_malignant"}
for c in ["cnv_arm_malignant", "tcr_clone_id", "tcr_is_dominant_clone", "tcr_is_expanded",
          "tcr_clone_size", "tcr_is_malignant"]:
    if c in _mp.columns:
        s = _mp[c].reindex(adata.obs_names)
        adata.obs[c] = (s.fillna(False).astype(bool).to_numpy() if c in _bool
                        else s.fillna("" if c == "tcr_clone_id" else 0).to_numpy())
del _al, _mp
mal = adata.obs["tcr_malignant_alice"].to_numpy()
adata.obs["is_malig"] = mal
print("malignant cells (TCR-ALICE):", int(mal.sum()),
      "| cnv_arm_malignant (nb30):", int(adata.obs["cnv_arm_malignant"].sum()))

# ---- cached arm-CNV matrix (nb30 v4) + its held-out healthy diploid null ----
# The v4 inferCNV pass relabels half the shared reference as `ref_null` and renames those rows
# `<run>|NULL|<cell>`, so they traverse the identical estimator as the tumour cells and ARE the
# empirical diploid null. nb31 used to discard them at the reindex below; they now calibrate the
# subclone gates (§A.1) and anchor the cohort arm-profile plot (§A figures).
_arm_raw = pd.read_parquet(ARM).set_index("obs_name")
ARM_COLS = S.arm_order([c for c in _arm_raw.columns if c != "donor"])
_arm_query, _null_arm = S.split_null_rows(_arm_raw)
NULL_ARMS = _null_arm[[*ARM_COLS, "ref_source", "donor"]]
NULL_X = NULL_ARMS[ARM_COLS].fillna(0.0).to_numpy(dtype=float)   # built once, reused per donor
del _arm_raw, _null_arm
arm_df = _arm_query.reindex(adata.obs_names)
print("arm matrix:", _arm_query.shape, "| healthy diploid null:", NULL_X.shape)
print(NULL_ARMS["ref_source"].value_counts().to_string())

# per-donor arm-CNV coverage (cache may omit donors that lacked a diploid reference for inferCNV).
cov_by_donor = (arm_df[ARM_COLS].notna().any(axis=1)
                .groupby(adata.obs["donor"].astype(str)).mean())

# ---- sample selection: eligibility for subclone detection (>= MIN_MAL malignant cells + arm CNV) ----
elig_tbl = (adata.obs.assign(_m=mal)
            .groupby("donor", observed=True)
            .agg(study=("study", "first"), disease=("disease", "first"),
                 n_cells=("_m", "size"), n_malignant=("_m", "sum"))
            .reset_index())
elig_tbl["n_malignant"] = elig_tbl["n_malignant"].astype(int)
elig_tbl["arm_cnv_cov"] = elig_tbl["donor"].map(lambda d: float(cov_by_donor.get(d, 0.0)))


def _exclude_reason(row):
    if str(row["disease"]) == "HC":
        return "healthy control — no malignant compartment"
    if row["n_malignant"] == 0:
        return "no ALICE-malignant cells (no dominant TCR clone)"
    if row["n_malignant"] < MIN_MAL:
        return f"<{MIN_MAL} malignant cells — too few for subclone KMeans"
    if row["arm_cnv_cov"] == 0.0:
        return "no arm-CNV coverage — donor absent from the inferCNV arm cache (nb30)"
    return ""


elig_tbl["reason_excluded"] = elig_tbl.apply(_exclude_reason, axis=1)
elig_tbl["eligible"] = elig_tbl["reason_excluded"].eq("")
elig_tbl = elig_tbl.sort_values(["eligible", "n_malignant"], ascending=[False, False]).reset_index(drop=True)
elig_tbl.to_csv(ELIG_CSV, index=False)

ELIG = sorted(elig_tbl.loc[elig_tbl["eligible"], "donor"])
n_arm_excl = int(elig_tbl["reason_excluded"].str.startswith("no arm-CNV").sum())
print(f"cohort donors: {len(elig_tbl)} | eligible (>= {MIN_MAL} malignant + arm CNV): {len(ELIG)}"
      f" | excluded: {int((~elig_tbl['eligible']).sum())} (of which {n_arm_excl} for 0 arm-CNV coverage)")
print("arms:", len(ARM_COLS), "| wrote", ELIG_CSV)

## §A.1 — Calculation (compute once, no plots)

In [ ]:
# Per-donor TCR founder + ALICE <=1-aa variant family (mirror nb30 PT35 exception).
PT35 = "Li2024_atlas__PT35"
mal_obs = adata.obs[mal].copy()
clono = A.clonotype_table(mal_obs, group="donor")
founder_sets, family_sets = {}, {}
for d, sub in clono.groupby("donor", observed=True):
    if d == PT35:
        seeds = set(sub.sort_values("n_cells", ascending=False)["cdr3"].iloc[:2])
    elif sub["is_founder"].astype(bool).any():
        seeds = set(sub.loc[sub["is_founder"].astype(bool), "cdr3"])
    elif len(sub):
        seeds = {sub.sort_values("n_cells", ascending=False)["cdr3"].iloc[0]}
    else:
        seeds = set()
    founder_sets[d] = seeds
    family_sets[d] = A.founder_family(sub, seeds=seeds) if seeds else set()
n_var = {d: len(family_sets[d] - founder_sets[d]) for d in founder_sets}
print("donors with >=1 ALICE <=1-aa TCR-beta variant:",
      sum(v > 0 for v in n_var.values()), "/", len(n_var))

In [ ]:
# per-donor CNV subclones (KMeans on the arm matrix, gates null-anchored) + summary + dominant subclone.
sub_label = pd.Series("", index=adata.obs_names, dtype=object)
summary_rows, centroids = [], {}
for d in ELIG:
    cells = adata.obs_names[mal & (adata.obs["donor"].astype(str).values == d)]
    Ad = arm_df.reindex(cells)[ARM_COLS]
    lab, meta = S.detect_subclones(Ad, k_max=K_MAX, min_sub=MIN_SUB, seed=SEED,
                                   sil_min=SIL_MIN, min_arm_delta=MIN_ARM_DELTA,
                                   null_arms=NULL_X, null_z=NULL_Z, null_draws=NULL_DRAWS)
    sub_label.loc[cells] = [f"{d}_s{c}" for c in lab]
    cen = meta["centroids"]; cen.index = [f"{d}_s{i}" for i in range(len(cen))]
    centroids[d] = cen
    summary_rows.append({"donor": d, "n_malig": len(cells), "k": meta["k"],
                         "silhouette": meta["silhouette"], "max_arm_delta": meta["max_arm_delta"],
                         "centroid_spread": meta["centroid_spread"],
                         "frac_cnv_confirmed": round(float(adata.obs.loc[cells, "cnv_arm_malignant"].mean()), 3),
                         "sizes": ",".join(map(str, meta["sizes"])), "n_tcr_variants": n_var.get(d, 0),
                         "sil_thr": meta["sil_thr"], "arm_delta_thr": meta["arm_delta_thr"],
                         "sil_margin": meta["sil_margin"], "null_calibrated": meta["null_calibrated"]})

adata.obs["cnv_subclone"] = sub_label.values
vc = adata.obs.loc[sub_label.values != "", "cnv_subclone"].value_counts()
dom_sub = {d: vc[[s for s in vc.index if s.rsplit("_s", 1)[0] == d]].idxmax() for d in ELIG}
adata.obs["is_dominant_subclone"] = adata.obs["cnv_subclone"].isin(set(dom_sub.values())).values

summary = pd.DataFrame(summary_rows)
meta_cols = ["study", "disease", "disease_stage", "entity"]
summary = summary.merge(adata.obs[["donor", *meta_cols]].astype(str).drop_duplicates("donor"),
                        on="donor", how="left")
n_multi = int((summary["k"] > 1).sum())
print(f">= 2 CNV subclones: {n_multi}/{len(summary)} = {n_multi/max(1,len(summary)):.0%} donors (paper: 84%)")
print(f"TCR/CNV concordance (mean frac_cnv_confirmed): {summary['frac_cnv_confirmed'].mean():.2f}")
print(f"gates: null-anchored (mean + {NULL_Z}sd of a size/k-matched diploid split) for "
      f"{int(summary['null_calibrated'].sum())}/{len(summary)} donors")
summary.sort_values("k", ascending=False)

In [ ]:
# trunk (pan-clonal, ancestral) vs branch (subclone-divergent) arm events per donor.
# Restored from the nb23 framework (Herrmann/Iyer 2025 §3); feeds §B.6 arm x activity crosses.
benign_mask = ((~adata.obs["tcr_is_dominant_clone"].to_numpy())
               & (adata.obs["cell_type"].astype(str).values != "tumor_cell"))
tb_rows = []
for d in ELIG:
    cen = centroids[d]
    dmask = adata.obs["donor"].astype(str).values == d
    A_all = arm_df.loc[adata.obs_names[mal & dmask], ARM_COLS].dropna(how="all")
    A_ben = arm_df.loc[adata.obs_names[benign_mask & dmask], ARM_COLS].dropna(how="all")
    if len(A_all) < 20 or len(A_ben) < 20:
        continue
    clone_events = S.call_arm_events(A_all, A_ben, z_thr=Z_THR, eps=EPS)
    tb = S.trunk_branch(clone_events, cen, branch_delta=BRANCH_DELTA)
    trunk_arms = ";".join(f"{a}{'+' if tb['trunk_signed'][a] > 0 else '-'}" for a in tb["trunk"])
    branch_arms = ";".join(
        f"{a}{'+' if cen[a].loc[cen[a].abs().idxmax()] > 0 else '-'}" for a in tb["branch"])
    row = summary.loc[summary["donor"] == d].iloc[0]
    tb_rows.append({"donor": d, "k": int(row["k"]), "trunk_arms": trunk_arms,
                    "branch_arms": branch_arms, "n_trunk": len(tb["trunk"]),
                    "n_branch": len(tb["branch"]), "study": row.get("study", ""),
                    "disease": row.get("disease", ""), "disease_stage": row.get("disease_stage", "")})
trunk_branch_df = pd.DataFrame(tb_rows)
trunk_branch_df.to_csv(TRUNKBRANCH_CSV, index=False)
print("wrote", TRUNKBRANCH_CSV, trunk_branch_df.shape)
trunk_branch_df.head()

In [ ]:
# transcriptomic MAJOR (Leiden on MRVI mu embedding) + CNV MINOR tracks -> nested subclones.
_lat = LATENT if LATENT in adata.obsm else ("X_scVI" if "X_scVI" in adata.obsm else "X_mrvi_u")
tracks, umap_by_donor, mm = S.major_minor_tracks(
    adata, _lat, mal, ELIG, cnv_col="cnv_subclone",
    leiden_res=LEIDEN_RES, n_neighbors=N_NEIGHBORS, seed=SEED)
for col in ["major_subclone", "minor_subclone", "subclone_label"]:
    adata.obs[col] = tracks[col].reindex(adata.obs_names).fillna("").astype(str).values
adata.obs["sub_umap1"] = tracks["sub_umap1"].reindex(adata.obs_names).values
adata.obs["sub_umap2"] = tracks["sub_umap2"].reindex(adata.obs_names).values
summary = summary.merge(mm[["donor", "n_major", "n_minor", "has_major", "has_minor",
                            "nmi_major_vs_minor"]], on="donor", how="left")
n_both = int((mm["has_major"] & mm["has_minor"]).sum())
print(f"MAJOR = Leiden (res={LEIDEN_RES}) on {_lat}; MINOR = CNV subclones "
      f"| clones with both tracks split: {n_both}/{len(mm)}")

In [ ]:
# cohort-wide NESTED subclones (transcriptomic MAJOR -> CNV MINOR within each major).
nested, split_summary = S.nested_subclone_labels(
    umap_by_donor, arm_df, ARM_COLS, ELIG,
    k_max=K_MAX, min_sub=MIN_SUB, seed=SEED, sil_min=SIL_MIN, min_arm_delta=MIN_ARM_DELTA,
    null_arms=NULL_X, null_z=NULL_Z, null_draws=NULL_DRAWS)
adata.obs["nested_subclone"] = nested.reindex(adata.obs_names).fillna("").astype(str).values
n_split = int((split_summary["k"] > 1).sum())
print(f"nested subclones: {nested.nunique()} across {len(ELIG)} eligible donors "
      f"| majors that split further on CNV: {n_split}/{len(split_summary)}")

In [ ]:
# malignant cells, log-normalized from raw counts (scoring + DE) + panels + dominant nested.
mal_ad = adata[adata.obs["cnv_subclone"].astype(str).values != ""].copy()
mal_ad.X = mal_ad.layers["raw_counts"].copy()
sc.pp.normalize_total(mal_ad, target_sum=1e4); sc.pp.log1p(mal_ad)
mal_ad.obs["nested_subclone"] = adata.obs.loc[mal_ad.obs_names, "nested_subclone"].astype(str).values

vc = mal_ad.obs["nested_subclone"].value_counts(); vc = vc[vc.index != ""]
by_donor = {}
for s, n in vc.items():
    by_donor.setdefault(s.rsplit("_", 1)[0], []).append((s, n))
dom_nested = {d: max(v, key=lambda t: t[1])[0] for d, v in by_donor.items()}
multi_nested = sorted(d for d, v in by_donor.items() if len(v) >= 2)
mal_ad.obs["is_dominant_nested"] = mal_ad.obs["nested_subclone"].isin(set(dom_nested.values())).values

panels = S.present_panels(mal_ad, S.PAPER_PANELS)
print("malignant cells for DE/scoring:", mal_ad.shape, "| donors:", mal_ad.obs["donor"].nunique())
print(f"nested subclones: {vc.shape[0]} | donors with >=2 nested subclones: {len(multi_nested)}")

In [ ]:
# functional-program scores + skin-layer tropism (nested axis) + persist.
added = S.score_programs(mal_ad, prefix="prog_", cell_cycle=True, seed=SEED)
prog_cols = [c for c in added if c.startswith("prog_")]

layer = mal_ad.obs["tissue"].astype(str).str.lower()
is_epi = layer.str.contains("epiderm"); is_derm = layer.str.contains("derm") & ~is_epi
mal_ad.obs["skin_layer"] = np.where(is_epi, "epidermis", np.where(is_derm, "dermis", "other"))
trop_tbl, cell_trop = S.classify_subclone_tropism(mal_ad.obs, "nested_subclone", min_cells=MIN_LAYER_CELLS)
mal_ad.obs["tropism"] = cell_trop.values

MENU = S.layer_split_donors(mal_ad.obs, "nested_subclone", min_cells=MIN_LAYER_CELLS)

adata.obs[["donor", "cnv_subclone", "nested_subclone", "is_dominant_subclone"]].to_parquet(SUBCLONE_PARQUET)
summary.to_csv(SUMMARY_CSV, index=False)
print(f"tropism classes: {trop_tbl['tropism'].value_counts().to_dict()}")
print("wrote", SUBCLONE_PARQUET.name, "+", SUMMARY_CSV.name)

In [ ]:
# Export gene signatures for downstream SPATIAL scoring (nb24/25 Visium).
import json
TOP_SUB, TOP_MAL = 30, 50
signatures = {}
for d in multi_nested:
    sub = mal_ad[mal_ad.obs["donor"].astype(str).values == d].copy()
    if sub.obs["nested_subclone"].nunique() < 2:
        continue
    de = S.subclone_markers_wilcoxon(sub, "nested_subclone", n_genes=sub.n_vars)
    de = de[(de["pval_adj"] < 0.05) & (de["log2fc"] > 0)]
    for s, g in de.sort_values("score", ascending=False).groupby("subclone", observed=True):
        genes = g["gene"].head(TOP_SUB).tolist()
        if len(genes) >= 5:
            signatures[f"subclone__{s}"] = genes

rng = np.random.default_rng(SEED)
m = adata.obs["is_malig"].to_numpy()
idx_m, idx_b = np.where(m)[0], np.where(~m)[0]
CAP = 20000
take = np.concatenate([rng.choice(idx_m, min(CAP, idx_m.size), replace=False),
                       rng.choice(idx_b, min(CAP, idx_b.size), replace=False)])
mvb = adata[take].copy()
mvb.X = mvb.layers["raw_counts"].copy()
sc.pp.normalize_total(mvb, target_sum=1e4); sc.pp.log1p(mvb)
mvb.obs["malig_grp"] = np.where(mvb.obs["is_malig"].to_numpy(), "malignant", "benign")
de_mal = S.subclone_markers_wilcoxon(mvb, "malig_grp", n_genes=mvb.n_vars)
de_mal = de_mal[(de_mal["subclone"] == "malignant") & (de_mal["pval_adj"] < 0.05) & (de_mal["log2fc"] > 0)]
signatures["malignant_overall"] = de_mal.sort_values("score", ascending=False)["gene"].head(TOP_MAL).tolist()
del mvb; gc.collect()

for cat, genes in S.PAPER_PANELS.items():
    signatures[f"panel__{cat}"] = list(dict.fromkeys(genes))
for prog, genes in S.PROGRAM_SETS.items():
    signatures[f"program__{prog}"] = list(dict.fromkeys(genes))

SIGN_JSON.write_text(json.dumps(signatures, indent=1))
print(f"wrote {SIGN_JSON.name}: {len(signatures)} signatures")

## §A — Cohort overview & per-sample figures

Cohort CNV-subclone arm profiles, prevalence of ≥2 subclones, program-score heatmap. For a per-sample
UMAP + arm-CNV view, set `SAMPLE` to any donor in `MENU` and call `S.plot_nested_sample`.

In [ ]:
print(f"{len(MENU)} samples with epidermis/dermis separation:")
print(MENU.to_string(index=False))
S.plot_cohort_arm_profiles(centroids, ELIG, ARM_COLS, null_arms=NULL_ARMS, vlim=Z_VLIM,
                           save=FIG_DIR / "subclone_v3_arm_profiles_cohort.png")
S.plot_subclone_prevalence(summary, k_max=K_MAX, n_multi=n_multi, save=FIG_DIR / "subclone_v3_prevalence.png")
S.plot_program_heatmap(mal_ad.obs, "nested_subclone", prog_cols,
                       donor_order=multi_nested, save=FIG_DIR / "subclone_v3_programs.png")

In [ ]:
# Per-sample subclone structure — pick ONE donor from MENU above.
SAMPLE = MENU["donor"].iloc[0] if len(MENU) else (ELIG[0] if ELIG else None)
if SAMPLE is not None:
    print("selected:", SAMPLE)
    S.plot_nested_sample(adata, SAMPLE, umap_by_donor, arm_df, ARM_COLS, mal, fig_dir=FIG_DIR)

# §B — TCR / co-stimulation signaling across malignant clones & subclones

Where in the TCR/costim cascade each malignant population concentrates dysregulation (module scores +
individual genes), vs lineage-matched reactive-CD4, cross-checked against arm-level inferCNV.
Reuses the in-memory object from §A; rebuilds `X` from `raw_counts` and normalizes the whole object.

In [ ]:
import warnings
import tcr_signaling_helpers as th
importlib.reload(th)

DATA = OUT_DIR
DE_DIR_S = DATA / "tcr_signaling_de"; DE_DIR_S.mkdir(exist_ok=True)
SUBCLONES_V3 = SUBCLONE_PARQUET
ARM_CNV_V3   = ARM
TRUNK_BRANCH_V3 = TRUNKBRANCH_CSV
MALIG_COL = "tcr_malignant_alice"
SUBCLONE_COL = "nested_subclone"
GROUPS = ("malignant", "reactive_CD4")

In [ ]:
# rebuild X from raw_counts on the full object, normalize+log1p (mirror nb26 §0)
try:
    del mal_ad
except NameError:
    pass
gc.collect()
adata.X = adata.layers["raw_counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print("normalized full object for signaling scoring:", adata.shape)

In [ ]:
# join subclone assignments + per-cell arm-CNV
sub = pd.read_parquet(SUBCLONES_V3)
if "cell_id" in sub.columns:
    sub = sub.set_index("cell_id")
for c in ["nested_subclone", "cnv_subclone", "is_dominant_subclone"]:
    if c in sub.columns:
        adata.obs[c] = sub[c].reindex(adata.obs_names)

arm = pd.read_parquet(ARM_CNV_V3)
if "obs_name" in arm.columns:
    arm = arm.set_index("obs_name")
ARM_COLS = [c for c in arm.columns if c.startswith("chr")]
arm = arm.reindex(adata.obs_names)
print("arm cnv coverage:", arm[ARM_COLS].notna().any(axis=1).mean())

In [ ]:
# groups: malignant vs lineage-matched reactive CD4
malig = adata.obs[MALIG_COL].astype("boolean").fillna(False).to_numpy()
ct = adata.obs["cell_type_T2"].astype(str)
grp = np.where(malig, "malignant",
               np.where((~malig) & (ct == "CD4"), "reactive_CD4", "other"))
adata.obs["malig_group"] = pd.Categorical(grp, categories=["malignant", "reactive_CD4", "other"])

tl = adata.obs["tissue"].astype(str).str.lower()
adata.obs["skin_layer"] = np.select(
    [tl.str.contains("epi"), tl.str.contains("derm")], ["epidermis", "dermis"], default="other")

print(adata.obs["malig_group"].value_counts())
if "stage_class" in adata.obs:
    print(adata.obs["stage_class"].value_counts())
print(adata.obs["skin_layer"].value_counts())

## §B.1 — Score modules (+ `_nolabile` sensitivity re-score)

In [ ]:
cov = th.module_coverage(adata)
try:
    display(cov)
except NameError:
    print(cov)

score_cols = th.score_tcr_modules(adata, seed=SEED, prefix="sig_")
score_cols_nl = th.score_tcr_modules(adata, seed=SEED, prefix="sig_nolabile_", drop=th.DISSOC_LABILE)
print(f"{len(score_cols)} module scores added")

mod_scores = adata.obs[score_cols].copy()
genes_all = sorted({g for gs in th.TCR_MODULES.values() for g in gs} & set(adata.var_names))
gene_expr = th._dense_gene_frame(adata, genes_all)
ACT_COLS = [f"sig_{m}" for m in th.ACTIVITY_MODULES if f"sig_{m}" in score_cols]
print(len(genes_all), "module genes present;", len(ACT_COLS), "activity modules")

## §B.2 — Within-sample: malignant vs reactive-CD4 (per donor)

In [ ]:
grp_s = adata.obs["malig_group"]
donor_s = adata.obs["donor"].astype(str)
keep = grp_s.isin(GROUPS)

mod_stats = th.wilcoxon_two_group(mod_scores[keep.values], score_cols, grp_s[keep], GROUPS,
                                  level="module", donor_series=donor_s[keep])
gene_stats = th.wilcoxon_two_group(gene_expr[keep.values], genes_all, grp_s[keep], GROUPS,
                                   level="gene", donor_series=donor_s[keep])
mod_stats["module"] = mod_stats["feature"].str.replace("sig_", "", regex=False)
mod_stats["context_only"] = mod_stats["module"].isin(th.CONTEXT_ONLY)
mod_stats.to_csv(DE_DIR_S / "within_sample_module_v3.csv", index=False)
gene_stats.to_csv(DE_DIR_S / "within_sample_gene_v3.csv", index=False)
mod_stats.head()

In [ ]:
# module x donor dotplot + clustered heatmap
plot_df = mod_stats.copy(); plot_df["feature"] = plot_df["module"]
fig, ax = th.module_score_dotplot(plot_df, group_col="donor", feature_col="feature",
                                  title="Malignant vs reactive-CD4 module scores (per donor)")
fig.savefig(FIG_DIR / "tcr_signaling_within_sample_module_dotplot.png", dpi=150, bbox_inches="tight")

def _first(s): return s.astype(str).iloc[0]
def _dom_layer(s):
    s = s[s.isin(["epidermis", "dermis"])]
    return s.value_counts().idxmax() if len(s) else np.nan

_ob = adata.obs.groupby("donor", observed=True)
donor_annot = pd.DataFrame({"study": _ob["study"].agg(_first),
                            "stage": _ob["stage_class"].agg(_first) if "stage_class" in adata.obs else "",
                            "layer": _ob["skin_layer"].agg(_dom_layer)})
STRIP_PALS = {"stage": {"early": "#4daf4a", "advanced": "#e41a1c"},
              "layer": {"epidermis": "#377eb8", "dermis": "#ff7f00"}}
cg = th.module_donor_clustermap(mod_stats, group_col="donor", feature_col="module",
                                value_col="cliffs_delta", col_annot=donor_annot,
                                annot_palettes=STRIP_PALS,
                                title="Malignant vs reactive-CD4 module Δ (donors clustered)")
cg.savefig(FIG_DIR / "tcr_signaling_within_sample_module_clustermap.png", dpi=150, bbox_inches="tight")

In [ ]:
# gene-level consistency across donors + per-family lollipops + activity-module violins
g = (gene_stats.assign(sig=gene_stats["fdr"] < 0.05)
     .groupby("feature")
     .agg(cliffs_delta=("cliffs_delta", "mean"), n_sig=("sig", "sum"), n_donor=("donor", "nunique"))
     .reset_index())
g["module"] = g["feature"].map(lambda x: th.GENE_TO_MODULE.get(x, [""])[0])
top = g.reindex(g["cliffs_delta"].abs().sort_values(ascending=False).index).head(40)
ax = th.lollipop_effects(top, effect="cliffs_delta", label="feature", fdr_col=None,
                         title="Top genes: malignant vs reactive-CD4 (mean Cliff's δ)", figsize=(5, 8))
ax.figure.savefig(FIG_DIR / "tcr_signaling_within_sample_gene_lollipop.png", dpi=150, bbox_inches="tight")

fam_figs = th.lollipop_per_family(g, effect="cliffs_delta", feature_col="feature", fdr_col=None,
                                  title_prefix="mal vs reactive-CD4: ")
for _m, _fig in fam_figs.items():
    _fig.savefig(FIG_DIR / f"tcr_signaling_within_sample_lollipop_{_m}.png", dpi=150, bbox_inches="tight")

obs_v = adata.obs.loc[keep.values, ["malig_group", *ACT_COLS]].copy()
for col in ACT_COLS:
    fig, _ = th.module_violin(obs_v, col, split_col="malig_group", order=list(GROUPS), title=col)
    fig.savefig(FIG_DIR / f"tcr_signaling_within_sample_violin_{col}.png", dpi=150, bbox_inches="tight")
print(len(fam_figs), "family lollipops;", len(ACT_COLS), "activity-module violins")

## §B.3 — Within malignant: each nested subclone vs its donor's reactive-CD4

In [ ]:
sub_series = adata.obs[SUBCLONE_COL].astype(str)
grp_all = adata.obs["malig_group"].astype(str)
don_all = adata.obs["donor"].astype(str)

subclone_mod = th.subclone_vs_control_stats(mod_scores, score_cols, sub_series, grp_all, don_all,
                                            level="module", control_group="reactive_CD4")
subclone_gene = th.subclone_vs_control_stats(gene_expr, genes_all, sub_series, grp_all, don_all,
                                             level="gene", control_group="reactive_CD4")
subclone_mod["module"] = subclone_mod["feature"].str.replace("sig_", "", regex=False)
subclone_mod["context_only"] = subclone_mod["module"].isin(th.CONTEXT_ONLY)
subclone_gene["module"] = subclone_gene["feature"].map(lambda x: th.GENE_TO_MODULE.get(x, [""])[0])
subclone_mod.to_csv(DE_DIR_S / "subclone_vs_control_module_v3.csv", index=False)
subclone_gene.to_csv(DE_DIR_S / "subclone_vs_control_gene_v3.csv", index=False)
print(subclone_mod["subclone"].nunique(), "subclones (module);",
      subclone_gene["subclone"].nunique(), "(gene) vs same-donor reactive-CD4")

In [ ]:
# cohort module x subclone dotplot + clustered heatmap + per-subclone gene volcanoes
if not subclone_mod.empty:
    n_sub = subclone_mod["subclone"].nunique()
    fig, ax = th.module_score_dotplot(subclone_mod, group_col="subclone", feature_col="module",
                                      title="Subclone vs reactive-CD4 module scores",
                                      figsize=(max(6, 0.28 * n_sub + 3), 8))
    fig.savefig(FIG_DIR / "tcr_signaling_subclone_module_dotplot.png", dpi=150, bbox_inches="tight")

    _gs = adata.obs[adata.obs[SUBCLONE_COL].astype(str) != ""].groupby(SUBCLONE_COL, observed=True)
    sub_meta = pd.DataFrame({"sample": _gs["donor"].agg(_first),
                             "study": _gs["study"].agg(_first),
                             "stage": _gs["stage_class"].agg(_first) if "stage_class" in adata.obs else "",
                             "layer": _gs["skin_layer"].agg(_dom_layer)})
    cg = th.module_donor_clustermap(subclone_mod, group_col="subclone", feature_col="module",
                                    value_col="cliffs_delta", col_annot=sub_meta,
                                    annot_palettes=STRIP_PALS, show_col_labels=False,
                                    title="Subclone vs reactive-CD4 module Δ (subclones clustered)",
                                    figsize=(max(9, 0.18 * n_sub + 4), 7))
    cg.savefig(FIG_DIR / "tcr_signaling_subclone_module_clustermap.png", dpi=150, bbox_inches="tight")

VOLC_DIR = FIG_DIR / "tcr_signaling_subclone_volcano"; VOLC_DIR.mkdir(exist_ok=True)
subs_all = sorted(subclone_gene["subclone"].unique()) if not subclone_gene.empty else []
for s in subs_all:
    d = subclone_gene[subclone_gene["subclone"] == s]
    ax = th.gene_volcano(d, effect="cliffs_delta", p_col="fdr", label="feature",
                         module_col="module", title=f"{s} vs reactive-CD4")
    ax.figure.savefig(VOLC_DIR / f"{s.replace('/', '_')}.png", dpi=150, bbox_inches="tight")
    plt.close(ax.figure)
print(len(subs_all), "subclone volcanoes ->", VOLC_DIR)

## §B.4 — Global + compartment: all malignant vs all reactive-CD4

In [ ]:
mod_pooled = th.wilcoxon_two_group(mod_scores[keep.values], score_cols, grp_s[keep], GROUPS,
                                   level="module", donor_series=None)
mod_pooled["module"] = mod_pooled["feature"].str.replace("sig_", "", regex=False)
mod_pooled["context_only"] = mod_pooled["module"].isin(th.CONTEXT_ONLY)
mod_pooled.to_csv(DE_DIR_S / "global_module_v3.csv", index=False)
ax = th.lollipop_effects(mod_pooled, label="module",
                         title="All malignant vs all reactive-CD4 (module Cliff's δ)", figsize=(5, 7))
ax.figure.savefig(FIG_DIR / "tcr_signaling_global_module_lollipop.png", dpi=150, bbox_inches="tight")

gene_pooled = th.wilcoxon_two_group(gene_expr[keep.values], genes_all, grp_s[keep], GROUPS,
                                    level="gene", donor_series=None)
gene_pooled["module"] = gene_pooled["feature"].map(lambda x: th.GENE_TO_MODULE.get(x, [""])[0])
gene_pooled.to_csv(DE_DIR_S / "global_gene_v3.csv", index=False)
act_genes = gene_pooled[gene_pooled["module"].isin(th.ACTIVITY_MODULES)]
ax = th.lollipop_effects(act_genes, label="feature",
                         title="Activity-module genes: malignant vs reactive-CD4", figsize=(5, 9))
ax.figure.savefig(FIG_DIR / "tcr_signaling_global_activity_gene_lollipop.png", dpi=150, bbox_inches="tight")

In [ ]:
# stratified by disease stage / skin layer
def stratified_module(strat_col, strata):
    out = []
    if strat_col not in adata.obs:
        return pd.DataFrame()
    for s in strata:
        m = keep.values & (adata.obs[strat_col].astype(str) == s).values
        if m.sum() < 50:
            continue
        r = th.wilcoxon_two_group(mod_scores[m], score_cols, grp_s[m], GROUPS,
                                  level="module", donor_series=None)
        r["stratum"] = s
        out.append(r)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

stage_stats = stratified_module("stage_class", ["early", "advanced"])
layer_stats = stratified_module("skin_layer", ["epidermis", "dermis"])
for df, name in [(stage_stats, "stage"), (layer_stats, "layer")]:
    if df.empty:
        continue
    df["feature"] = df["feature"].str.replace("sig_", "", regex=False)
    fig, ax = th.module_score_dotplot(df, group_col="stratum", feature_col="feature",
                                      title=f"Malignant vs reactive-CD4 by {name}")
    fig.savefig(FIG_DIR / f"tcr_signaling_by_{name}_module_dotplot.png", dpi=150, bbox_inches="tight")

## §B.5 — Signaling-mode localization per subclone

In [ ]:
mal = adata[malig].copy()
mal_donor = mal.obs["donor"].astype(str)
contrast_df = th.subclone_contrast_vector(mal.obs, SUBCLONE_COL, score_prefix="sig_")
contrast_df = contrast_df[contrast_df.index.astype(str).str.len() > 0]
g2 = th.signaling_mode_heatmap(contrast_df, title="Subclone signaling-mode contrasts")
g2.savefig(FIG_DIR / "tcr_signaling_mode_heatmap.png", dpi=150, bbox_inches="tight")

from scipy.cluster.hierarchy import fcluster, linkage
contrast_names = [c for c in th.CONTRASTS if c in contrast_df.columns]
cvec = contrast_df[contrast_names].fillna(0.0)
if len(cvec) >= 4:
    Z = linkage(cvec, method="ward")
    contrast_df["mode_cluster"] = fcluster(Z, t=4, criterion="maxclust")
    names = {}
    for cl, subc in contrast_df.groupby("mode_cluster"):
        mc = subc[contrast_names].mean()
        names[cl] = f"{mc.abs().idxmax()}{'+' if mc[mc.abs().idxmax()] > 0 else '-'}"
    contrast_df["mode_label"] = contrast_df["mode_cluster"].map(names)
    print(contrast_df["mode_label"].value_counts())

## §B.6 — arm-CNV × downstream activity cross

In [ ]:
tb = pd.read_csv(TRUNK_BRANCH_V3)
arm_events = th.parse_arm_events(tb)
sub_means = mal.obs.groupby(SUBCLONE_COL, observed=True)[ACT_COLS].mean()
sub_means["donor"] = mal.obs.groupby(SUBCLONE_COL, observed=True)["donor"].agg(
    lambda x: x.astype(str).iloc[0])
sub_means = sub_means[sub_means.index.astype(str).str.len() > 0].reset_index()

for armn, direction, module, note in th.ARM_ACTIVITY_MAP:
    col = f"sig_{module}"
    if col not in sub_means.columns:
        continue
    fig, ax = th.arm_activity_box(sub_means, arm_events, armn, direction, module,
                                  title=f"{armn}{direction} -> {module}\n{note}")
    fig.savefig(FIG_DIR / f"tcr_signaling_cnv_{armn}{direction}_{module}.png", dpi=150, bbox_inches="tight")

# support: per-donor Spearman(continuous arm score, matched activity module) over malignant cells
from scipy.stats import spearmanr
rows = []
arm_mal = arm.reindex(mal.obs_names)
for armn, direction, module, _ in th.ARM_ACTIVITY_MAP:
    acol = f"sig_{module}"
    if armn not in arm_mal.columns or acol not in mal.obs:
        continue
    for d in mal_donor.unique():
        m = (mal_donor == d).values
        x = arm_mal[armn].to_numpy()[m]; y = mal.obs[acol].to_numpy()[m]
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() < 30:
            continue
        rho, p = spearmanr(x[ok], y[ok])
        rows.append({"donor": d, "arm": armn, "direction": direction, "module": module,
                     "rho": rho, "p": p, "n": int(ok.sum())})
arm_corr = pd.DataFrame(rows)
arm_corr.to_csv(DE_DIR_S / "arm_activity_correlation_v3.csv", index=False)
print("arm x activity correlations:", arm_corr.shape)

## §B.7 — Sensitivity (IEG dissociation) & save

In [ ]:
labile_mods = [m for m in ["L4_ieg_acute", "L3c_ap1_mapk_activity"] if f"sig_{m}" in adata.obs]
comp = []
for m in labile_mods:
    full = th.wilcoxon_two_group(mod_scores[keep.values], [f"sig_{m}"], grp_s[keep], GROUPS,
                                 level="module", donor_series=None)
    nl = th.wilcoxon_two_group(adata.obs.loc[keep, [f"sig_nolabile_{m}"]], [f"sig_nolabile_{m}"],
                               grp_s[keep], GROUPS, level="module", donor_series=None)
    comp.append({"module": m, "cliffs_full": full["cliffs_delta"].iloc[0],
                 "cliffs_nolabile": nl["cliffs_delta"].iloc[0]})
print(pd.DataFrame(comp))

out = contrast_df.reset_index().rename(columns={"index": SUBCLONE_COL})
out.to_csv(DATA / "tcr_signaling_subclone_modes_v3.csv", index=False)
print("wrote", DATA / "tcr_signaling_subclone_modes_v3.csv")
print("DE tables in", DE_DIR_S, "| figures in", FIG_DIR)

# §C — Machine-readable numeric summary

One flat `KEY: value` dump of every number this notebook produces: object/compartment counts,
donor eligibility, per-donor CNV subclones (k, silhouette, TCR/CNV concordance), MAJOR/MINOR and
nested subclones, trunk vs branch arm events, layer tropism, exported signatures, and for §B the
group sizes, module coverage, per-donor and pooled malignant-vs-reactive-CD4 effects, subclone
contrasts, signaling-mode clusters, arm-CNV × activity correlations and the IEG sensitivity check —
plus an index of the figures/tables on disk.

Run **last** (it reuses the in-memory objects from §A and §B). Blocks whose inputs are missing are
skipped. Also written to `tables/subclone_tcr_signaling_summary.txt`.

In [ ]:
# --- §C · Machine-readable summary: every number this notebook reports, in one dump ---
# Flat `KEY: value` lines covering §A (subclonal structure) and §B (TCR/costim signaling).
# Written to tables/subclone_tcr_signaling_summary.txt so an agent can read the result
# without re-running the heavy object.
_LOG = []


def P(s=""):
    _LOG.append(str(s)); print(s)


def SEC(t):
    P(); P("=" * 78); P(f"## {t}"); P("=" * 78)


def kv(k, v):
    P(f"{k}: {v}")


def pct(a, b):
    return f"{100 * a / b:.2f}%" if b else "n/a"


def has(name):
    return name in globals() and globals()[name] is not None


def top_effects(df, label_col, n=10, effect="cliffs_delta", fdr_col="fdr"):
    """Rank features by |effect|; return printable lines (sign = direction vs control)."""
    if df is None or len(df) == 0 or effect not in df:
        return ["   (empty)"]
    agg = {effect: (effect, "mean")}
    if fdr_col in df:
        agg["frac_fdr05"] = (fdr_col, lambda s: round(float((s < 0.05).mean()), 2))
    if "donor" in df:
        agg["n_donor"] = ("donor", "nunique")
    if "subclone" in df:
        agg["n_subclone"] = ("subclone", "nunique")
    g = df.groupby(label_col, observed=True).agg(**agg).reset_index()
    g = g.reindex(g[effect].abs().sort_values(ascending=False).index).head(n)
    return [f"   {str(r[label_col])[:38]:<38} " +
            " ".join(f"{c}={r[c]:.3g}" for c in g.columns if c != label_col)
            for _, r in g.iterrows()]


TAB_DIR = NB_DIR / "tables"; TAB_DIR.mkdir(exist_ok=True)
obs = adata.obs
N = adata.n_obs

SEC("0 · PROVENANCE & PARAMETERS")
kv("notebook", "31_subclone_tcr_signaling.ipynb — subclonal evolution + TCR/costim signaling")
kv("object", str(OBJ))
kv("malignancy_definition",
   "tcr_malignant_alice (nb30 ALICE dominant founder + <=1-aa TRB-CDR3 variant family)")
kv("inputs", f"{ALICE_MAL.name}, {MALIG_PARQUET.name}, {ARM.name}")
kv("outputs", f"{SUBCLONE_PARQUET.name}, {SUMMARY_CSV.name}, {TRUNKBRANCH_CSV.name}, "
              f"{ELIG_CSV.name}, {SIGN_JSON.name}")
kv("params", f"SEED={SEED} MIN_MAL={MIN_MAL} K_MAX={K_MAX} MIN_SUB={MIN_SUB} "
             f"SIL_MIN={SIL_MIN} MIN_ARM_DELTA={MIN_ARM_DELTA} NULL_Z={NULL_Z} NULL_DRAWS={NULL_DRAWS} "
             f"Z_THR={Z_THR} EPS={EPS} BRANCH_DELTA={BRANCH_DELTA} MIN_LAYER_CELLS={MIN_LAYER_CELLS}")
kv("major_track", f"Leiden res={LEIDEN_RES}, n_neighbors={N_NEIGHBORS} on {LATENT}")
kv("minor_track", "KMeans on the 41-arm inferCNV matrix (nb30 skin_T_arm_cnv_v4)")
kv("subclone_gates", f"null-anchored: mean + {NULL_Z} x sd of a size/k-matched KMeans split of "
                     f"{NULL_X.shape[0]:,} held-out healthy diploid cells ({NULL_DRAWS} draws); "
                     f"SIL_MIN/MIN_ARM_DELTA are the null-less fallback.")

SEC("1 · OBJECT & COMPARTMENT")
kv("n_cells", f"{N:,}")
kv("n_genes", f"{adata.n_vars:,}")
kv("n_donors", obs["donor"].nunique())
kv("n_studies", obs["study"].nunique())
P("\n-- T subtype (cell_type_T2) --")
for k, v in obs["cell_type_T2"].astype(str).value_counts().items():
    P(f"   {k:<14} n={v:>8,} ({100 * v / N:5.1f}%)")
_mal = obs["is_malig"].to_numpy().astype(bool)
_cnv = obs["cnv_arm_malignant"].to_numpy().astype(bool)
kv("malignant_ALICE(tcr_malignant_alice)", f"{int(_mal.sum()):,} ({pct(int(_mal.sum()), N)})")
kv("cnv_arm_malignant(nb30)", f"{int(_cnv.sum()):,} ({pct(int(_cnv.sum()), N)})")
kv("TCR_and_CNV_overlap", f"{int((_mal & _cnv).sum()):,} | jaccard="
                          f"{(_mal & _cnv).sum() / max(1, (_mal | _cnv).sum()):.3f}")
kv("cells_with_arm_CNV_coverage", pct(int(arm[ARM_COLS].notna().any(axis=1).sum()), N)
   if has("arm") else "n/a")
kv("donors_with_ALICE_1aa_variant_family",
   f'{sum(v > 0 for v in n_var.values())}/{len(n_var)}' if has("n_var") else "n/a")

SEC("2 · DONOR ELIGIBILITY FOR SUBCLONE DETECTION")
kv("cohort_donors", len(elig_tbl))
kv("eligible(>=MIN_MAL malignant)", f'{int(elig_tbl["eligible"].sum())} '
   f'({pct(int(elig_tbl["eligible"].sum()), len(elig_tbl))})')
P("\n-- exclusion reasons --")
for k, v in elig_tbl.loc[~elig_tbl["eligible"], "reason_excluded"].value_counts().items():
    P(f"   {k:<52} n_donors={v}")
P("\n-- eligible donors (malignant cells) --")
P(elig_tbl.loc[elig_tbl["eligible"], ["donor", "study", "disease", "n_cells", "n_malignant"]]
  .to_string(index=False))

SEC("3 · CNV SUBCLONES PER DONOR (§A.1)")
kv("donors_analyzed", len(summary))
kv("donors_with>=2_subclones", f'{int((summary["k"] > 1).sum())}/{len(summary)} = '
   f'{(summary["k"] > 1).mean():.0%}  (paper reference: 84%)')
P("\n-- k distribution --")
for k, v in summary["k"].value_counts().sort_index().items():
    P(f"   k={k}: {v} donors")
kv("silhouette_median", f'{summary["silhouette"].median():.3f}')
kv("max_arm_delta_median", f'{summary["max_arm_delta"].median():.3f}')
kv("frac_cnv_confirmed_mean(TCR/CNV concordance)", f'{summary["frac_cnv_confirmed"].mean():.3f}')
kv("n_tcr_variants_total", int(summary["n_tcr_variants"].sum()))
P("\n-- per-donor summary --")
P(summary.sort_values("k", ascending=False).to_string(index=False))

if has("mm"):
    SEC("4 · MAJOR (transcriptomic) / MINOR (CNV) / NESTED SUBCLONES")
    kv("clones_split_on_both_tracks", f'{int((mm["has_major"] & mm["has_minor"]).sum())}/{len(mm)}')
    kv("n_major_median", f'{mm["n_major"].median():.0f}')
    kv("n_minor_median", f'{mm["n_minor"].median():.0f}')
    kv("nmi_major_vs_minor_median", f'{mm["nmi_major_vs_minor"].median():.3f}'
       if "nmi_major_vs_minor" in mm else "n/a")
    kv("n_nested_subclones", int(nested.nunique()))
    kv("majors_that_split_further_on_CNV",
       f'{int((split_summary["k"] > 1).sum())}/{len(split_summary)}')
    if has("multi_nested"):
        kv("donors_with>=2_nested_subclones", f"{len(multi_nested)}: {', '.join(multi_nested)}")
    if has("vc"):
        P("\n-- nested subclone sizes (cells) --")
        P(vc.to_string())

if has("trunk_branch_df") and len(trunk_branch_df):
    SEC("5 · TRUNK vs BRANCH ARM EVENTS (§A.1)")
    kv("donors_with_trunk_branch_call", len(trunk_branch_df))
    kv("n_trunk_median/max", f'{trunk_branch_df["n_trunk"].median():.0f} / '
                             f'{trunk_branch_df["n_trunk"].max():.0f}')
    kv("n_branch_median/max", f'{trunk_branch_df["n_branch"].median():.0f} / '
                              f'{trunk_branch_df["n_branch"].max():.0f}')
    kv("donors_with_any_branch_event", int((trunk_branch_df["n_branch"] > 0).sum()))
    for col, lab in [("trunk_arms", "trunk (ancestral, pan-clonal)"),
                     ("branch_arms", "branch (subclone-divergent)")]:
        ev_counts = (trunk_branch_df[col].fillna("").str.split(";").explode()
                     .replace("", np.nan).dropna().value_counts())
        P(f"\n-- most recurrent {lab} arm events (n_donors) --")
        for k, v in ev_counts.head(12).items():
            P(f"   {k:<10} {v}")
    P("\n-- per donor --")
    P(trunk_branch_df.to_string(index=False))

if has("trop_tbl"):
    SEC("6 · SKIN-LAYER TROPISM OF NESTED SUBCLONES")
    kv("min_cells_per_layer", MIN_LAYER_CELLS)
    for k, v in trop_tbl["tropism"].value_counts().items():
        P(f"   {k:<20} n_subclones={v}")
    if has("MENU"):
        kv("donors_with_epidermis/dermis_split", len(MENU))
        P(MENU.to_string(index=False))

if has("signatures"):
    SEC("7 · EXPORTED GENE SIGNATURES (for spatial scoring, nb24/25)")
    kv("file", str(SIGN_JSON))
    kv("n_signatures", len(signatures))
    for pref in ["subclone__", "panel__", "program__", "malignant_overall"]:
        keys = [k for k in signatures if k.startswith(pref)]
        kv(f"n_{pref.rstrip('_')}", f"{len(keys)} "
           f"(median size {int(np.median([len(signatures[k]) for k in keys]))})" if keys else "0")
    kv("malignant_overall_top20", ", ".join(signatures.get("malignant_overall", [])[:20]))

SEC("8 · SIGNALING GROUPS (§B)")
P("-- malig_group (foreground vs lineage-matched control) --")
for k, v in obs["malig_group"].value_counts().items():
    P(f"   {k:<14} n={v:>8,} ({100 * v / N:5.1f}%)")
P("-- skin_layer --")
for k, v in obs["skin_layer"].astype(str).value_counts().items():
    P(f"   {k:<14} n={v:>8,}")
if "stage_class" in obs:
    P("-- stage_class --")
    for k, v in obs["stage_class"].astype(str).value_counts().items():
        P(f"   {k:<14} n={v:>8,}")
kv("n_module_scores", len(score_cols))
kv("activity_modules", ", ".join(c.replace("sig_", "") for c in ACT_COLS))
kv("module_genes_present", len(genes_all))
if has("cov"):
    P("\n-- module gene coverage (present/total) --")
    P(cov[["module", "n_present", "n_total", "context_only"]].to_string(index=False))

SEC("9 · WITHIN-SAMPLE: malignant vs reactive-CD4, per donor (§B.2)")
kv("donors_tested", mod_stats["donor"].nunique() if len(mod_stats) else 0)
kv("effect", "Cliff's delta, positive = higher in malignant; frac_fdr05 = fraction of donors FDR<0.05")
P("\n-- top modules (mean over donors) --")
for l in top_effects(mod_stats, "module", n=14):
    P(l)
P("\n-- top genes (mean over donors) --")
for l in top_effects(gene_stats, "feature", n=20):
    P(l)

if has("mod_pooled"):
    SEC("10 · GLOBAL POOLED: all malignant vs all reactive-CD4 (§B.4)")
    P("-- modules --")
    P(mod_pooled[["module", "cliffs_delta", "p", "fdr", "direction"]]
      .reindex(mod_pooled["cliffs_delta"].abs().sort_values(ascending=False).index)
      .to_string(index=False))
    if has("gene_pooled"):
        P("\n-- top 25 genes --")
        gp = gene_pooled.reindex(
            gene_pooled["cliffs_delta"].abs().sort_values(ascending=False).index).head(25)
        P(gp[["feature", "module", "cliffs_delta", "fdr", "direction"]].to_string(index=False))

if has("subclone_mod") and len(subclone_mod):
    SEC("11 · SUBCLONE vs SAME-DONOR reactive-CD4 (§B.3)")
    kv("n_subclones_module_level", subclone_mod["subclone"].nunique())
    kv("n_subclones_gene_level",
       subclone_gene["subclone"].nunique() if has("subclone_gene") else 0)
    P("\n-- modules ranked by mean effect across subclones --")
    for l in top_effects(subclone_mod, "module", n=14):
        P(l)
    P("\n-- per-subclone strongest module --")
    best = (subclone_mod.reindex(subclone_mod["cliffs_delta"].abs()
                                 .sort_values(ascending=False).index)
            .drop_duplicates("subclone")
            .sort_values("subclone"))
    P(best[["subclone", "donor", "module", "cliffs_delta", "fdr"]].to_string(index=False)
      if "donor" in best else best[["subclone", "module", "cliffs_delta", "fdr"]].to_string(index=False))

for _df, _name in [(stage_stats if has("stage_stats") else None, "stage_class"),
                   (layer_stats if has("layer_stats") else None, "skin_layer")]:
    if _df is None or _df.empty:
        continue
    SEC(f"12 · STRATIFIED by {_name} (§B.4)")
    for s, d in _df.groupby("stratum", observed=True):
        d2 = d.reindex(d["cliffs_delta"].abs().sort_values(ascending=False).index).head(6)
        P(f"-- {s} (n_modules={len(d)}) --")
        P(d2[["feature", "cliffs_delta", "fdr"]].to_string(index=False))

if has("contrast_df") and len(contrast_df):
    SEC("13 · SIGNALING-MODE LOCALIZATION PER SUBCLONE (§B.5)")
    kv("n_subclones", len(contrast_df))
    kv("contrasts", ", ".join(f"{k}={v[0]}-{v[1]}" for k, v in th.CONTRASTS.items()))
    if "mode_label" in contrast_df:
        P("\n-- dominant signaling mode (ward, 4 clusters) --")
        for k, v in contrast_df["mode_label"].value_counts().items():
            P(f"   {k:<32} n_subclones={v}")
    _cn = [c for c in th.CONTRASTS if c in contrast_df.columns]
    P("\n-- contrast values per subclone --")
    P(contrast_df[_cn + [c for c in ["mode_label"] if c in contrast_df]]
      .round(2).to_string())

if has("arm_corr") and len(arm_corr):
    SEC("14 · arm-CNV x DOWNSTREAM ACTIVITY (§B.6)")
    kv("mapping", "; ".join(f"{a}{d}->{m}" for a, d, m, _ in th.ARM_ACTIVITY_MAP))
    kv("n_donor_x_arm_tests", len(arm_corr))
    ac = (arm_corr.groupby(["arm", "direction", "module"], observed=True)
          .agg(n_donors=("donor", "nunique"), median_rho=("rho", "median"),
               frac_p05=("p", lambda s: round(float((s < 0.05).mean()), 2)),
               median_n_cells=("n", "median"))
          .reset_index().sort_values("median_rho", key=abs, ascending=False))
    P(ac.to_string(index=False))

if has("comp") and len(comp):
    SEC("15 · SENSITIVITY: IEG/dissociation-labile genes dropped (§B.7)")
    kv("interpretation", "cliffs_nolabile close to cliffs_full => effect not driven by "
                         "dissociation-induced IEGs")
    P(pd.DataFrame(comp).to_string(index=False))

SEC("16 · ARTIFACT INDEX")
P(f"figures ({FIG_DIR}):")
for f in sorted(FIG_DIR.glob("subclone_v3*.png")) + sorted(FIG_DIR.glob("tcr_signaling*.png")):
    P(f"   {f.name:<58} {f.stat().st_size / 1e3:7.0f} kB")
if has("VOLC_DIR") and VOLC_DIR.exists():
    kv("subclone_volcanoes", f"{len(list(VOLC_DIR.glob('*.png')))} png in {VOLC_DIR}")
P(f"tables ({DE_DIR_S}):")
for f in sorted(DE_DIR_S.glob("*.csv")):
    P(f"   {f.name:<58} {f.stat().st_size / 1e3:7.0f} kB")
P("data outputs:")
for f in [SUBCLONE_PARQUET, SUMMARY_CSV, TRUNKBRANCH_CSV, ELIG_CSV, SIGN_JSON,
          DATA / "tcr_signaling_subclone_modes_v3.csv"]:
    P(f"   {f.name:<58} exists={f.exists()}")

_TXT = TAB_DIR / "subclone_tcr_signaling_summary.txt"
_TXT.write_text("\n".join(_LOG))
print(f"\n[summary written -> {_TXT}]")